# Export `financial_data` to NDJSON

Produces the file-based counterpart of `financial_data` so that a Spark variant
can run without MongoDB in the loop.

`companies` needs no export: the Brreg bulk download `enheter_alle.json`
already is the file, and the profiling pass confirmed the raw file and the
MongoDB collection hold an identical set of 64 top-level fields.
`financial_data` has no file equivalent, because it was assembled from ~1.17M
individual Regnskapsregisteret API calls. **This export is a reconstruction,
not the original ingestion path.** A genuinely file-native pipeline would have
written each API response to disk as it arrived and never involved MongoDB.
That variant would also have hit the small-files problem, since 1.17M
individual responses is pathological for distributed processing. Re-fetching to
demonstrate that is not practical at 1 req/s, so the report should describe
this file as equivalent in content but not in provenance.

NDJSON rather than a single JSON array, for three reasons: Spark's writer emits
only NDJSON, so an array would require collecting 1.17M documents through the
driver; an array is not splittable on read, so the financial side would become
single-threaded like the companies side; and appending one JSON object per line
is what an incremental fetch loop actually produces. The asymmetry against
`enheter_alle.json` is therefore realistic rather than a flaw: a bulk download
and an incremental fetch genuinely differ in framing. **Writes are staged
through container-local disk.** Spark's commit renames fail intermittently
against Dropbox, and they fail after `mode("overwrite")` has deleted the
previous export. Spark writes to `/tmp` inside the container and the finished
files are copied onto the mount, so Dropbox no longer has to be paused for a
write on the order of a gigabyte. See `staged_write.py`.

In [1]:
import json
import os

from pyspark.sql import SparkSession

from schemas import COMPANIES_SCHEMA, FINANCIAL_SCHEMA
from staged_write import write_staged

MONGO_DB = "companiesdb"
DATA_DIR = "/home/jovyan/data"
PARQUET_DIR = os.path.join(DATA_DIR, "parquet")
NDJSON_DIR = os.path.join(DATA_DIR, "ndjson")
RAW_COMPANIES = os.path.join(DATA_DIR, "enheter_alle.json")

# Driver memory, thread count, the Mongo connector package and the connection
# URI all come from jupyter/spark-defaults.conf, which is baked into the image.
# Nothing about the JVM is configured here, so the notebook cannot drift from
# the environment a grader gets. Change the config file and rebuild instead.
spark = SparkSession.builder.appName("group13_ndjson_export").getOrCreate()

_conf = spark.sparkContext.getConf()
MONGO_URI = _conf.get("spark.mongodb.read.connection.uri")
CONNECTOR = _conf.get("spark.jars.packages")

print("Spark        ", spark.version)
print("master       ", spark.sparkContext.master)
print("driver heap  %.1f GB" % (spark._jvm.java.lang.Runtime.getRuntime().maxMemory() / 1024**3))
print("connector    ", CONNECTOR)
print("mongo uri    ", MONGO_URI)
print("host cores   ", os.cpu_count())

Spark         4.2.0
master        local[4]
driver heap  8.0 GB
connector     org.mongodb.spark:mongo-spark-connector_2.13:11.1.0
mongo uri     mongodb://mongodb:27017
host cores    12


## Change detection

Same signature approach as the Parquet export. A full rewrite takes minutes, so
it is skipped when the source is unchanged. Merge logic would add failure modes
to save a few minutes at this size.

In [2]:
from datetime import datetime, timezone

from pymongo import MongoClient

METADATA_PATH = os.path.join(NDJSON_DIR, "_export_metadata.json")

# Set True to re-export even when the source signature is unchanged.
FORCE_REFRESH = False

client = MongoClient(MONGO_URI)
db = client[MONGO_DB]

newest = db.financial_data.find_one(sort=[("fetched_at", -1)],
                                    projection={"fetched_at": 1})
current = {
    "count": db.financial_data.count_documents({}),
    "max_fetched_at": newest["fetched_at"].isoformat() if newest and newest.get("fetched_at") else None,
}

previous = {}
if os.path.exists(METADATA_PATH):
    with open(METADATA_PATH) as fh:
        previous = json.load(fh).get("signature", {})

stale = FORCE_REFRESH or current != previous
print(json.dumps(current, indent=2))
print("\nfinancial_data  %s" % ("export needed" if stale else "unchanged, skipping"))

{
  "count": 1170292,
  "max_fetched_at": "2026-09-03T12:40:15.842000"
}

financial_data  unchanged, skipping


## Export

Read through the connector with the full `FINANCIAL_SCHEMA`, including the
nested `data` statement blob, then write as NDJSON.

The write is repartitioned to `defaultParallelism`. Without it the connector's
own partitioning propagates through, which produced an uneven file layout; with
it the reader gets one split per core. This is a deliberate advantage handed to
the NDJSON variant, and the benchmark notes it: the raw `companies` file gets
no such help, because it arrives as a single unsplittable array and nothing can
be done about that without rewriting it.

In [3]:
os.makedirs(NDJSON_DIR, exist_ok=True)
TARGET = os.path.join(NDJSON_DIR, "financial_data")

if stale:
    df = (
        spark.read.format("mongodb")
        .option("database", MONGO_DB)
        .option("collection", "financial_data")
        .schema(FINANCIAL_SCHEMA)
        .load()
    )
    # Staged through container-local disk, then copied onto the mount: Spark's
    # commit renames fail intermittently against Dropbox, and they fail after
    # mode("overwrite") has deleted the previous export. See staged_write.py.
    write_staged(df.repartition(spark.sparkContext.defaultParallelism),
                 TARGET, "json")
    # Count from the written files rather than the DataFrame, which would
    # otherwise re-read the whole collection from MongoDB a second time.
    rows = spark.read.schema(FINANCIAL_SCHEMA).json(TARGET).count()
    print("wrote %d rows" % rows)

    # Written only after a successful export, so an interrupted run stays
    # marked stale rather than falsely up to date.
    with open(METADATA_PATH, "w") as fh:
        json.dump({"signature": current,
                   "exported_at": datetime.now(timezone.utc).isoformat(),
                   "rows_written": rows}, fh, indent=2)
    print("Metadata updated.")
else:
    print("Skipped.")

Skipped.


## Verification

Two things are checked. Row count against MongoDB, and a round-trip of the
statement values.

The round-trip matters because Spark serialises `fetched_at` as an ISO-8601
string in JSON and must parse it back as a timestamp. If that fails the column
silently becomes null, and the benchmark would compare a working Parquet read
against a broken NDJSON read.

In [4]:
from pyspark.sql import functions as F

back = spark.read.schema(FINANCIAL_SCHEMA).json(TARGET)

rows = back.count()
mongo_rows = db.financial_data.count_documents({})
print("rows  ndjson=%d  mongo=%d  %s"
      % (rows, mongo_rows, "OK" if rows == mongo_rows else "MISMATCH"))
assert rows == mongo_rows, "ndjson/mongo row mismatch"

# Nulls here would mean a parse failure, not missing source data.
checks = back.select(
    F.sum(F.col("fetched_at").isNull().cast("int")).alias("null_fetched_at"),
    F.sum(F.col("data").isNotNull().cast("int")).alias("with_statement"),
    F.sum(F.col("data")[0]["resultatregnskapResultat"]["aarsresultat"]
          .isNotNull().cast("int")).alias("with_aarsresultat"),
    F.sum(F.col("data")[0]["resultatregnskapResultat"]["totalresultat"]
          .isNotNull().cast("int")).alias("with_totalresultat"),
).collect()[0]

# Measured from MongoDB in the same run rather than compared against a stored
# figure. Mirrors Spark's isNotNull: missing and explicit null both count as null.
# "$ne: None" excludes both missing fields and explicit nulls, matching Spark's
# isNotNull. ("missing" is only a valid type alias in the aggregation $type
# expression, not in a query.)
mongo_with_data = db.financial_data.count_documents({"data": {"$ne": None}})

print("\n%-30s %12s %12s" % ("metric", "ndjson", "mongo"))
print("-" * 58)
print("%-30s %12d %12d %s" % ("records with data", checks["with_statement"],
                              mongo_with_data,
                              "OK" if checks["with_statement"] == mongo_with_data
                              else "MISMATCH"))
assert checks["with_statement"] == mongo_with_data, "ndjson/mongo data mismatch"

# A non-zero count here means a parse failure, which is true at any corpus size
# and needs no reference value.
print("\nnull fetched_at:      %d   (must be 0)" % checks["null_fetched_at"])
assert checks["null_fetched_at"] == 0, "fetched_at failed to parse"

# Reported only: array-element null handling is the least obviously equivalent
# part of the two engines, so a difference is information, not a failure.
print("with aarsresultat:    %d" % checks["with_aarsresultat"])
print("with totalresultat:   %d" % checks["with_totalresultat"])

size = sum(os.path.getsize(os.path.join(d, f))
           for d, _, files in os.walk(TARGET) for f in files)
print("\nNDJSON on disk: %.2f GB in %d files"
      % (size / 1024**3, len(os.listdir(TARGET))))

rows  ndjson=1170292  mongo=1170292  OK

metric                               ndjson        mongo
----------------------------------------------------------
records with data                    444646       444646 OK

null fetched_at:      0   (must be 0)
with aarsresultat:    444646
with totalresultat:   241560

NDJSON on disk: 0.69 GB in 10 files
